# Lecture 2: The Ingestion Bottleneck and Streaming

## 1. The Shuffle Buffer Mechanism
A global Spark `reduceByKey` across a remote 10TB S3 bucket is computationally unfeasible due to memory constraints.

Let dataset size be $D$ and buffer size be $K$. True uniform random sampling requires $\mathcal{O}(D)$ memory.
However, a sliding-window buffer shuffling approach reduces this to $\mathcal{O}(K)$ memory, yielding pseudo-randomness sufficient for Stochastic Gradient Descent (SGD).


In [ ]:
import webdataset as wds
import matplotlib.pyplot as plt
import numpy as np

# We incorporate wds.shuffle into our pipeline
url = "http://127.0.0.1:9000/cifar-streaming/cifar-train-{000000..000049}.tar"

# Smaller buffer = less random uniform distribution, but uses less RAM
buffer_size = 5000  # Try changing this from 10 to 5000 and see the histogram change

dataset = (
    wds.WebDataset(url, shardshuffle=True)
    .shuffle(buffer_size)
    .decode("rgb")
    .to_tuple("png", "cls")
)

## 2. Optimizing the PyTorch DataLoader
The neural network is merely a consumer; the data engineer's job is to guarantee it is fed fast enough. We construct a `DataLoader` to optimize `batch_size`, `num_workers` (parallel downloading), and `pin_memory` (GPU page-locking).

In [ ]:
import torch
import time
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=256,
    num_workers=4,       # Experiment with 0 to 8 to find the saturation point
    pin_memory=True      # Allows fast transfer to CUDA Devices
)

# Simulate a training loop measuring Samples / Second
print("Starting benchmark loop...")
start = time.time()
samples = 0

try:
    for i, (images, labels) in enumerate(loader):
        # Simulate forward/backward pass processing time
        time.sleep(0.01)
        
        samples += images.size(0)
        if i == 4: # Run 5 minibatches
            break

    elapsed = time.time() - start
    print(f"Throughput: {samples / elapsed:.2f} Samples / Second")
except Exception as e:
    print(f"Could not read items. Is the local MinIO stream online? {e}")

## 3. Model Architecture Deep Dive: Small ResNet
Transitioning from MapReduce topologies to deep neural nets.
We define a residual mapping for the ResNet architecture:
$$y = \mathcal{F}(x, \{W_i\}) + x$$

The derivative allows gradient flow to bypass the non-linear blocks without vanishing:
$$\frac{\partial \mathcal{L}}{\partial x} = \frac{\partial \mathcal{L}}{\partial y} \left( \frac{\partial \mathcal{F}}{\partial x} + 1 \right)$$


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ResNet9(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        
        # Residual Block
        self.res_conv1 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.res_conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(64 * 16 * 16, 10)

    def forward(self, x):
        # Base Path
        out = F.relu(self.bn1(self.conv1(x)))
        
        # Residual Mapping: F(x) + x
        res = self.res_conv1(out)
        res = F.relu(res)
        res = self.res_conv2(res)
        out = F.relu(out + res) # The Addition
        
        out = self.pool(out)
        out = out.view(out.size(0), -1)
        return self.fc(out)

# Instantiate the model
model = ResNet9()
print(model)

## 4. The Consumer: PyTorch Lightning Training Loop
We standardize the training loop to abstract away device-casting and optimizer stepping.
We compute Categorical Cross-Entropy:
$$\mathcal{L} = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)$$


In [ ]:
import pytorch_lightning as pl
import torch.optim as optim

class TrafficLight(pl.LightningModule):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, labels = batch
        # Standard format
        if images.dtype == torch.uint8: # Webdataset might decode to 0-255 uint8 byte arrays
            images = images.float() / 255.0
            
        # Ensure channel first for PyTorch (N, C, H, W)
        if images.shape[-1] == 3: # (N, H, W, C) from WDS decode('rgb')
            images = images.permute(0, 3, 1, 2)
            
        outputs = self(images)
        loss = F.cross_entropy(outputs, labels)
        self.log('train_loss', loss)
        return loss

    def configure_optimizers(self):
        return optim.AdamW(self.model.parameters(), lr=1e-3)

# Note: In a real run, you'd trigger this using pl.Trainer(max_epochs=2).fit(TrafficLight(model), loader)
print("LightningModule defined! Ensure you use '%tensorboard --logdir lightning_logs' to track loss.")